# AI assistants: how they work and how they fail

This week is different. There is no new Python syntax. Instead, you learn
to work with the tools that now write much of the code in the world: **AI
assistants**. Just as important, you learn how *not* to be fooled by them.

Work through this notebook from top to bottom and run the code cells. You
also need a browser tab with the university chat assistant (see "Try it"
below).

This lesson is about **chat assistants**: you type, they answer with text.
Lesson 2 is about **agents**, which can also run commands on a computer.

## Coding has changed

Most code today is written *with* an AI assistant, not typed from memory.
The important skills are now:

- **saying** clearly what you want,
- **reading** and **running** what you get back,
- **checking** that it is actually correct.

That is why the last six weeks matter. You cannot check code that you
cannot read. Variables, loops, functions and types (weeks 1–4) let you read
the code. The command line and git (weeks 5–6) let you see and undo what
an agent does (lesson 2).

Think of the assistant as a fast, confident junior colleague. It is often
right, sometimes wrong, and it never tells you which. **You** sign off on
the result.

## What is a large language model?

The assistants are built on a **large language model** (LLM). The useful
idea, in one sentence:

> An LLM is a program that, given some text, predicts which text is likely
> to come next.

It learned this from a huge amount of text: much of the internet, books,
and program code. It learned the *patterns* in that text.

- It is **not** a database and **not** a search engine. It does not look up
  facts. It *generates* the most likely continuation.
- This one idea explains most of what it does well, and most of what it
  does badly.

## A tiny language model you can run

The cell below is a very small "language model". From a few sentences it
learns which word tends to follow which. Then it writes new text, one word
at a time. You do not need to understand every line. Run it a few times.

In [ ]:
import random

training_text = (
    "the penguin eats fish . the penguin swims in the sea . "
    "the seal eats the penguin . the seal swims fast . "
    "the fish swims in the sea ."
)
words = training_text.split()

# For each word, collect the words that follow it in the training text.
follows = {}
for i in range(len(words) - 1):
    word = words[i]
    next_one = words[i + 1]
    if word not in follows:
        follows[word] = []
    follows[word].append(next_one)

print("Words that follow 'penguin':", follows["penguin"])

# Write a new "sentence": start with 'the', then pick a likely next word.
word = "the"
sentence = [word]
for step in range(10):
    word = random.choice(follows[word])
    sentence.append(word)

print(" ".join(sentence))

Notice:

- It writes *plausible* sentences that were not in the training text. It
  recombines patterns.
- It has no idea what a penguin *is*. It only knows which words go
  together.
- Each run gives a different result, because it picks the next word at
  random among the likely ones.

A real LLM does the same job, predict the next piece of text, but it is
billions of times larger and much better at using the context of the whole
conversation.

## A few words you will hear

- **Token**: models read text in small pieces ("pen", "guin"), not whole
  words. Limits and prices are counted in tokens.
- **Context window**: how much text the model can "see" at once. In a very
  long chat, it loses the beginning. Start a new chat for a new task.
- **Randomness**: like the tiny model above, a real model picks among
  likely continuations. So **the same question can give different
  answers**.

## The one thing to remember

An LLM produces *plausible-looking text*. So it also produces
plausible-looking code and plausible-looking function names, whether or not
they are correct, or even exist.

This is not a bug that a better question can remove. It is how the tool
works. That is why most of this lesson is about **checking**, not
trusting.

## The loop: ask, read, run, test

Whether you use a chat or an agent, work in this loop:

1. **Ask**: say exactly what you want (inputs, output, an example).
2. **Read** the code you get. Does it do what you asked?
3. **Run** it. An error message is information, not failure.
4. **Test** it with an input where *you* know the answer.
5. **Repeat**: paste the error or the wrong result back, and ask for a fix.

## Asking well

Ask for one small, clear piece at a time. Compare:

**Weak:**

> write code to analyse my data

**Strong:**

> Write a Python function `mean_mass(masses)`. `masses` is a list of body
> masses in grams. Return the mean as a float. Leave out values of `-999`,
> which mean "not measured". Show one example call.

The strong question names the input, the output, a special case (`-999`),
and asks for an example. You get code that you can check.

Other useful questions:

- "Explain this code line by line." (Very good for learning.)
- "Here is the error message: ... What does it mean?"
- "Give me three test cases for this function, with the expected results."

## Three ways generated code goes wrong

From easiest to hardest to notice.

### 1. A function that does not exist

The model invents a function that *sounds* right. Here it suggested
`statistics.average`. Run the cell.

In [ ]:
import statistics

values = [1, 2, 3]
print(statistics.average(values))

It stops with `AttributeError: module 'statistics' has no attribute
'average'`. The real function is called `statistics.mean`.

**How you notice:** you run it, and Python complains loudly. This is the
easy case.

In [ ]:
print(statistics.mean(values))

### 2. Plausible, but the logic is wrong

The code runs without an error and returns something reasonable. But it is
wrong. This function should return the **fraction** of values above a
threshold:

In [ ]:
def fraction_above(values, threshold):
    count = 0
    for v in values:
        if v > threshold:
            count = count + 1
    return count / threshold

# Of [1, 2, 3, 4, 5], two values (4 and 5) are above 3: the answer should be 2/5 = 0.4
print(fraction_above([1, 2, 3, 4, 5], 3))

It prints `0.666...`, not `0.4`. It divides by `threshold` instead of by
the number of values, `len(values)`. No error, just a wrong number.

**How you notice:** test it with a case where you know the answer.

### 3. Silent data errors

The most dangerous kind. The code is correct in general, but wrong for
*your* data, and nothing complains. Real data sets often mark missing
values with a special number such as `-999`.

In [ ]:
# Penguin body masses in grams. -999 means "not measured".
masses = [3200, 4500, 3800, -999, 4100, -999, 3900]

print("mean of everything:", sum(masses) / len(masses))

measured = []
for m in masses:
    if m != -999:
        measured.append(m)
print("mean of the measured values:", sum(measured) / len(measured))

Both numbers look like the mass of a penguin. Only the second one is right.

**How you notice:** *look at your data*. Print the smallest and largest
values, or sort them. Read what the columns mean and how missing values are
written. You will meet this trap with real penguin data in the exercise.

## Before you trust generated code

- **Read** it. If you cannot explain it, you cannot trust it.
- **Run** it, with inputs whose answer you know.
- **Try special cases**: an empty list, missing values, lower-case letters.
- **Look at the data** first: how many rows, the smallest and largest
  values, the units.
- Prefer **small pieces** that you can check over one big block.

## You own what you submit

- The same question can give different code tomorrow. Write down which
  model you used and your important questions. This is part of doing
  reproducible science.
- Using AI assistants is allowed in this course. You must be able to
  understand and explain everything you submit, no matter who or what wrote
  it. The rules are in the course README (section "AI tools and working
  together").
- Never paste passwords, API keys (lesson 2), or personal data (for
  example data about patients or other students) into an AI tool.

## Try it: the university chat assistant

The university runs a chat assistant for its members:
https://openwebui.uni-freiburg.de . Log in with your university account.

At the top of the page you can choose a model. Choose a coding model, for
example "UFR: Standard Coding". The computing centre changes which model is
behind such a name from time to time.

Try two things:

1. Ask it to explain this error message, which you saw above:
   `AttributeError: module 'statistics' has no attribute 'average'`.
2. Paste the function `fraction_above` from above and ask: "This should
   return the fraction of values above the threshold. Is it correct?" Does
   it find the bug? Ask the same question again in a new chat. Is the
   answer the same?

## Summary

- An LLM predicts likely text. So it writes likely-looking code, which is
  often right and sometimes wrong.
- Work in the loop: ask, read, run, test, repeat.
- Three ways code goes wrong: functions that do not exist (Python
  complains), wrong logic (test with known answers), silent data errors
  (look at the data).
- You must be able to explain everything you submit.

## Going further (optional)

- Ask the chat assistant to write `fraction_above` for you. Then test its
  version with `[1, 2, 3, 4, 5]` and `3`.
- Change `training_text` in the tiny language model, for example to a few
  sentences from your favourite book. Which sentences does it produce now?
- Ask the same question to two different models. Which answer is better?
  How can you tell?

## Check yourself

Answer these without running anything first.

1. An assistant writes `max(['900', '1000', '3500'])` to find the largest
   body mass. What does it return, and why is that wrong?
2. You ask the same question twice and get two different functions. Is
   one of them wrong? How do you find out?
3. An assistant's code for the mean body mass runs without error and
   prints `4080.07`. Why is that not enough to trust it?

Answers are at the end of this notebook.

## Answers to check yourself

1. It returns `'900'`. The values are strings, and strings are compared
   letter by letter: `'9'` comes after `'3'` and `'1'`. Convert them to
   numbers first, for example with `int()`.
2. Not necessarily: there are many correct ways to write a function. Read
   both, and test both with inputs where you know the answer.
3. Code that runs can still be wrong for your data, for example if `-999`
   values were included. Look at the data (smallest, largest values) and
   check the result against something you know.